# Task 15: Hough Transform for Line Detection

Build a line-detection pipeline for synthetic images. You should be familiar with NumPy arrays and basic image coordinates. You will preprocess RGB images, detect edges, find lines, clip them to the image, and draw an overlay.

## 1. Libraries and Synthetic Images

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

axes_image = np.zeros((160, 200, 3), dtype=np.uint8)
axes_image[15:145, 48:52] = 255
axes_image[78:82, 12:188] = 255

geometry = np.zeros_like(axes_image)
cv2.rectangle(geometry, (35, 28), (164, 131), (255, 255, 255), 3)
cv2.line(geometry, (19, 140), (180, 14), (255, 255, 255), 3)

rng = np.random.default_rng(15)
broken = rng.integers(0, 35, size=axes_image.shape, dtype=np.uint8)
for start in (12, 72, 132):
    broken[58:62, start : start + 44] = 210

examples = {
    "Crossing axes": axes_image,
    "Rectangle and diagonal": geometry,
    "Interrupted line and noise": broken,
}

## 2. Hough Transform Fundamentals

Canny identifies likely boundary pixels. The Hough transform groups edge pixels that support the same infinite line through voting.

A line satisfies `rho = x * cos(theta) + y * sin(theta)`. The origin is at the top left, with x increasing rightward and y downward. The signed distance rho is measured along the normal from the origin, and theta is the angle of that normal in radians. Thus x = c has parameters (c, 0), and y = c has parameters (c, pi/2).

Each edge pixel votes along a curve in parameter space. Curves from collinear points intersect near the same parameters, forming a peak in the accumulator. Rho and angular resolutions set the bin widths. Coarser bins can merge nearby candidates, while finer bins can spread votes across several bins. Increasing the vote threshold generally suppresses weaker peaks.

In [ ]:
angles = np.linspace(0, np.pi, 361)
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
for x in (20, 70, 120):
    ax.plot(
        np.rad2deg(angles), x * np.cos(angles) + 40 * np.sin(angles), label=f"({x}, 40)"
    )
ax.scatter(90, 40, color="black", zorder=4, label="Shared parameters")
ax.set(xlabel="Theta (degrees)", ylabel="Rho (pixels)", title="Votes for y = 40")
ax.legend()
plt.show()

## 3. Edge Detection

Convert RGB to grayscale with the RGB conversion flag, apply a Gaussian blur such as 5×5, and run Canny with the supplied thresholds. Return a 2D uint8 edge map with values 0 and 255.

In [ ]:
def detectar_bordas(imagem_rgb, limiar_inferior=50, limiar_superior=150):
    """Edge Detection."""
    ### START CODE HERE ###
    # TODO: Edge Detection.
    raise NotImplementedError("Implement detectar_bordas.")
    ### END CODE HERE ###

## 4. Hough Line Detection

Use cv2.HoughLines with all three supplied parameters. The standard transform returns infinite parametric lines. HoughLinesP returns segments and is outside this exercise. Normalize the result to a floating-point (N, 2) array, including (0, 2) when no line is found.

In [ ]:
def detectar_retas(bordas, resolucao_rho=1.0, resolucao_theta=np.pi / 180, limiar=50):
    """Hough Line Detection."""
    ### START CODE HERE ###
    # TODO: Hough Line Detection.
    raise NotImplementedError("Implement detectar_retas.")
    ### END CODE HERE ###

## 5. Line Coordinate Conversion

The reference point is (rho cos(theta), rho sin(theta)), and the direction vector is (-sin(theta), cos(theta)). Extend in both directions using a distance based on the image dimensions and |rho|. Clip to the pixel rectangle [0, width−1] × [0, height−1]. You may use cv2.clipLine after computing the endpoints. Return two integer (x, y) points in either order, or None if the line misses the image.

In [ ]:
def calcular_extremos_reta(rho, theta, largura, altura):
    """Line Coordinate Conversion."""
    ### START CODE HERE ###
    # TODO: Line Coordinate Conversion.
    raise NotImplementedError("Implement calcular_extremos_reta.")
    ### END CODE HERE ###

## 6. Drawing Detected Lines

Copy the RGB image, reuse calcular_extremos_reta for each line, skip invisible lines, and draw the remaining ones with cv2.line using the supplied RGB color and thickness. Return an independent copy even for empty detections. Preserve all input arrays.

In [ ]:
def desenhar_retas(imagem_rgb, retas, cor=(255, 0, 0), espessura=2):
    """Drawing Detected Lines."""
    ### START CODE HERE ###
    # TODO: Drawing Detected Lines.
    raise NotImplementedError("Implement desenhar_retas.")
    ### END CODE HERE ###

## 7. Deterministic Tests

Run after completing the four TODOs. Tests use known geometry and allow equivalent Hough parameters, nearby detections, and arbitrary ordering.

In [ ]:
uniform = np.zeros((80, 100, 3), dtype=np.uint8)
contrast = uniform.copy()
contrast[:, 50:] = 255
for image, expect_edges in ((uniform, False), (contrast, True)):
    original = image.copy()
    edges = detectar_bordas(image)
    assert edges.shape == image.shape[:2] and edges.dtype == np.uint8
    assert np.isin(edges, [0, 255]).all()
    assert bool(edges.any()) == expect_edges
    np.testing.assert_array_equal(image, original)


def contains_line(lines, rho, theta):
    return any(
        np.any(
            (np.abs(lines[:, 0] - sign * rho) <= 2)
            & (np.abs(lines[:, 1] - theta - shift * np.pi) <= np.deg2rad(2))
        )
        for shift, sign in ((-1, -1), (0, 1), (1, -1))
    )


vertical = np.zeros((80, 100), dtype=np.uint8)
horizontal = vertical.copy()
diagonal = vertical.copy()
vertical[:, 30] = 255
horizontal[40, :] = 255
np.fill_diagonal(diagonal, 255)
for edge_map, rho, theta in (
    (vertical, 30, 0),
    (horizontal, 40, np.pi / 2),
    (diagonal, 0, 3 * np.pi / 4),
):
    original = edge_map.copy()
    lines = detectar_retas(edge_map, 1.0, np.pi / 180, 50)
    assert isinstance(lines, np.ndarray) and lines.ndim == 2
    assert lines.shape[1] == 2 and np.issubdtype(lines.dtype, np.floating)
    assert contains_line(lines, rho, theta)
    np.testing.assert_array_equal(edge_map, original)

empty_lines = detectar_retas(np.zeros_like(vertical))
assert isinstance(empty_lines, np.ndarray) and empty_lines.shape == (0, 2)
assert np.issubdtype(empty_lines.dtype, np.floating)


for rho, theta, expected, tolerance in (
    (30, 0, ((30, 0), (30, 79)), 0),
    (40, np.pi / 2, ((0, 40), (99, 40)), 0),
    (0, 3 * np.pi / 4, ((0, 0), (79, 79)), 1),
):
    endpoints = calcular_extremos_reta(rho, theta, 100, 80)
    assert endpoints is not None
    assert len(endpoints) == 2
    for x, y in endpoints:
        assert isinstance(x, (int, np.integer)) and isinstance(y, (int, np.integer))
        assert 0 <= x < 100 and 0 <= y < 80
    np.testing.assert_allclose(
        sorted(map(tuple, endpoints)), expected, atol=tolerance, rtol=0
    )

assert calcular_extremos_reta(110, 0, 100, 80) is None

drawing_input = uniform.copy()
drawing_lines = np.array([[30.0, 0.0], [110.0, 0.0]])
lines_before = drawing_lines.copy()
color = (12, 85, 230)
overlay = desenhar_retas(drawing_input, drawing_lines, cor=color, espessura=1)
expected = uniform.copy()
expected[:, 30] = color
assert overlay.shape == uniform.shape and overlay.dtype == uniform.dtype
np.testing.assert_array_equal(overlay, expected)
np.testing.assert_array_equal(drawing_input, uniform)
np.testing.assert_array_equal(drawing_lines, lines_before)
copy = desenhar_retas(drawing_input, empty_lines)
np.testing.assert_array_equal(copy, uniform)
assert not np.shares_memory(copy, drawing_input)
print("Test passed!")

## 8. Visual Comparison

Classical Hough lines can extend beyond the segments that supplied the votes. Canny can pick up both sides of a thick stroke, so nearby detections can be expected.

In [ ]:
def show_pipeline(
    image,
    title,
    canny_low=50,
    canny_high=150,
    rho_resolution=1.0,
    theta_degrees=1.0,
    votes=50,
):
    edges = detectar_bordas(image, canny_low, canny_high)
    lines = detectar_retas(edges, rho_resolution, np.deg2rad(theta_degrees), votes)
    overlay = desenhar_retas(image, lines)
    fig, panels = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
    fig.suptitle(f"{title} ({len(lines)} detected lines)")
    panels[0].imshow(image)
    panels[1].imshow(edges, cmap="gray", vmin=0, vmax=255)
    panels[2].imshow(overlay)
    for panel, label in zip(
        panels,
        ("Original image", "Canny edge map", "Detected Hough lines"),
        strict=True,
    ):
        panel.set_title(label)
        panel.axis("off")
    plt.show()


for title, image in examples.items():
    show_pipeline(image, title)

## 9. Parameter Experimentation

Compare at least three configurations on the same image. Record the settings, detection counts and plots, then explain any false positives or missed lines. Angular resolution is entered in degrees and converted by the helper.

In [ ]:
configurations = (
    dict(canny_low=30, canny_high=90, rho_resolution=1, theta_degrees=1, votes=20),
    dict(canny_low=50, canny_high=150, rho_resolution=1, theta_degrees=1, votes=50),
    dict(canny_low=50, canny_high=150, rho_resolution=2, theta_degrees=2, votes=80),
)
for index, settings in enumerate(configurations, start=1):
    show_pipeline(broken, f"Configuration {index}", **settings)

## 10. Results and Discussion

1. Why do collinear edge pixels vote for the same Hough parameter pair?
2. How do the Canny thresholds affect false positives and missed lines?
3. What happens when the Hough bins become wider or the vote threshold increases?
4. Why can a single thick stroke produce several detected lines?
5. How does a classical Hough overlay differ from a segment visualization?